# Local projections: incertidumbre comercial y actividad manufacturera por rama

Lee la base integrada (`data/processed/base_incertidumbre_comercial_mexico.xlsx`), construye la **sorpresa del TPU** y estima *local projections* (LP) en panel para **producción real, empleo y horas trabajadas**, con y sin el periodo de la pandemia.

**Especificación revisada (2026-10-01).** Cambia respecto a la primera versión en seis puntos; la sección 7 compara ambas.

| # | Antes | Ahora | Por qué |
|---|---|---|---|
| 1 | Variable dependiente en variación a 12 meses, Δ12 y(i,t+h) | **Cambio acumulado** log y(i,t+h) − log y(i,t−1) | Δ12 en el horizonte h mezcla meses anteriores al choque y, desde h = 12, resta la respuesta del año anterior (efecto base). |
| 2 | Efectos fijos de rama (y de mes en A) | Efectos fijos de **rama × mes del calendario** (y de mes en A) | Con el cambio acumulado hay que quitar la estacionalidad propia de cada rama. |
| 3 | En A, sin tipo de cambio ni VIX | En A, **Δ12 TC × E y Δ12 VIX × E** rezagados | Los efectos de mes absorben el tipo de cambio común, no su efecto diferencial sobre las ramas exportadoras. |
| 4 | Sin efectos de calendario | **Días hábiles** (en B; en A los absorben los efectos de mes) | Semana Santa y la composición de días cambian de un año a otro. |
| 5 | Errores Driscoll-Kraay y bootstrap por bloques | **Inferencia por permutación de la sorpresa** (Driscoll-Kraay solo como referencia) | Con sorpresas falsas, Driscoll-Kraay rechaza en cerca de 40% de los casos al 90% (debería ser 10%); la sección 6 lo documenta. |
| 6 | Sin placebos; pandemia: se excluía si t o t+h caía en ella | **Placebos** h = −12 a −5; se excluye si **cualquier mes** entre t−1 y t+h cae en la pandemia | Probar que las ramas expuestas no divergían antes del choque; la ventana completa entra en la variable dependiente. |

**Ecuaciones** (h = −12, …, −5 y 0, …, 12; i = rama, t = mes; c(t) = mes del calendario):

**A. Exposición diferencial** (principal):

log y(i,t+h) − log y(i,t−1) = a_{i,c(t)} + d_t + β_h · s_t × E_i + Σ_{l=1..3} [ρ'_l Δ log Y(i,t−l) + θ_l s_{t−l} × E_i + φ_l Δ12 TC_{t−l} × E_i + ψ_l Δ12 VIX_{t−l} × E_i] + Σ_{l=0..3} [γ_l Δ12 arancel(i,t−l) + κ_l Δ12 IP_EEUU_{t−l} × E_i] + e

**B. Efecto promedio** (descriptiva: se identifica solo con la variación en el tiempo de ~70-90 meses):

log y(i,t+h) − log y(i,t−1) = a_{i,c(t)} + β_h · s_t + Σ_{l=1..3} [ρ'_l Δ log Y(i,t−l) + θ_l s_{t−l} + φ_l Δ12 TC_{t−l} + ψ_l Δ12 VIX_{t−l}] + Σ_{l=0..3} [γ_l Δ12 arancel(i,t−l) + κ_l Δ12 IP_EEUU_{t−l}] + δ · Δ log días hábiles(t+h, t−1) + e

- **s_t** = sorpresa del TPU estandarizada (1 = una desviación estándar 2018-2026). **E_i** = coeficiente de exportación de la MIP 2018 (decisión D9). **Y** = las tres variables (producción, empleo, horas): sus variaciones mensuales rezagadas entran como control en todas las regresiones.
- **Lectura:** en A, diferencia en % entre una rama de exposición alta (p75) y una baja (p25) ante una sorpresa de 1 desviación estándar; en B, % para la rama promedio. Como la variable dependiente es el cambio acumulado, β_h es el efecto sobre el **nivel** h meses después.
- **Placebos (h < 0):** la "respuesta" de meses anteriores al choque. Si el diseño es válido, deben ser cero. h = −4 a −1 no se estiman porque coinciden casi por completo con los rezagos que van de control.

El notebook **no** modifica la base integrada.

In [1]:
import html, platform, warnings
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
from IPython.display import HTML, Markdown, display

warnings.filterwarnings("ignore")
def _raiz():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "data" / "processed").is_dir() and (q / "docs").is_dir():
            return q
    raise RuntimeError("No se encontró la raíz del repositorio.")
RAIZ = _raiz()
display(HTML("""<style>
.tabla{font-size:82%;border-collapse:collapse;margin-bottom:8px}.tabla th{background:#dce6f1;border:1px solid #bbb;padding:3px 6px;text-align:left}
.tabla td{border:1px solid #ddd;padding:2px 6px}.irf-grid{display:grid;grid-template-columns:repeat(2,minmax(0,430px));gap:10px 16px}
</style>"""))

def tabla(df, titulo=None, max_filas=120, dec=3):
    if titulo:
        display(Markdown(f"**{titulo}**"))
    d = df.head(max_filas).copy()
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].map(lambda v: "NA" if pd.isna(v) else f"{v:,.{dec}f}")
    display(HTML(d.to_html(index=False, border=0, classes="tabla", escape=True)))

def aviso(texto, tipo="info"):
    c = {"info": ("#e8f1fb", "#2c5aa0"), "revisar": ("#fff4e0", "#b26a00"), "ok": ("#e7f5ec", "#1b7f3b")}[tipo]
    display(HTML(f'<div style="background:{c[0]};border-left:6px solid {c[1]};padding:8px 12px;margin:8px 0">{texto}</div>'))

base = pd.read_excel(RAIZ / "data/processed/base_incertidumbre_comercial_mexico.xlsx", sheet_name="base_integrada", dtype={"rama": str, "subsector": str})
base["mes"] = pd.to_datetime(base.mes)
assert not base.duplicated(["rama", "mes"]).any()
assert "eeuu_ip_total" in base, "La base no tiene eeuu_ip_total: vuelva a ejecutar 01_preparacion_base_integrada.ipynb"
display(Markdown(f"**Base:** {base.shape[0]:,} filas, {base.rama.nunique()} ramas, {base.mes.min():%Y-%m} a {base.mes.max():%Y-%m}."))

**Base:** 8,858 filas, 86 ramas, 2018-01 a 2026-07.

## 1. Transformaciones
**Variaciones a 12 meses** (para los controles): diferencia de logaritmos para variables positivas; diferencia simple (en puntos) para tasas o índices que pueden valer cero. **Logaritmos y variaciones mensuales** de producción, empleo y horas (para la variable dependiente y sus rezagos). Todas las comparaciones entre meses se hacen por fecha, no por posición, porque el panel tiene huecos.

In [2]:
base = base.sort_values(["rama", "mes"]).reset_index(drop=True)
transformaciones = [
    ("emim_valor_produccion_real", "d12_produccion", "log", "Producción real (EMIM)"),
    ("emim_personal_ocupado", "d12_empleo", "log", "Personal ocupado (EMIM)"),
    ("emim_horas_trabajadas", "d12_horas", "log", "Horas trabajadas (EMIM)"),
    ("tpu", "d12_tpu", "log", "TPU (Caldara et al.)"),
    ("eeuu_ip_total", "d12_ip_eeuu", "log", "Producción industrial total de EE.UU."),
    ("eeuu_imp_mx_tasa_efectiva", "d12_arancel", "dif_pp", "Tasa arancelaria efectiva de EE.UU. a la rama"),
    ("tc_fix", "d12_tc", "log", "Tipo de cambio FIX"),
    ("vix_promedio_mensual", "d12_vix", "log", "VIX (promedio mensual)"),
    ("wtui_usa", "d12_wtui_usa", "dif", "WTUI EE.UU."),
    ("wtui_mundo_pib", "d12_wtui_mundo", "dif", "WTUI mundo"),
    ("wtui_mex", "d12_wtui_mex", "dif", "WTUI México"),
    ("epu_mex", "d12_epu", "log", "EPU México"),
]
anterior = base[["rama", "mes"]].assign(mes=base.mes + pd.DateOffset(years=1))
filas = []
for orig, nueva, metodo, desc in transformaciones:
    x = base[orig].astype(float)
    if metodo == "log":
        assert (x.dropna() > 0).all(), orig
        x = np.log(x)
    elif metodo == "dif_pp":
        x = 100 * x
    base = base.merge(anterior.assign(previo=x.values), on=["rama", "mes"], how="left", validate="one_to_one")
    base[nueva] = x.values - base.pop("previo")
    filas.append(dict(variable=nueva, original=orig, descripcion=desc,
                      metodo={"log": "Δ log", "dif": "diferencia en puntos del índice", "dif_pp": "diferencia en puntos porcentuales"}[metodo],
                      observaciones=int(base[nueva].notna().sum()), primer_mes=base.mes[base[nueva].notna()].min().strftime("%Y-%m")))
tabla(pd.DataFrame(filas), "Variables transformadas")

def en_fecha(col, k):
    """Valor de `col` en el mes t + k de la misma rama (NaN si ese mes no existe)."""
    aux = base[["rama", "mes", col]].assign(mes=base.mes - pd.DateOffset(months=k))
    return base[["rama", "mes"]].merge(aux, on=["rama", "mes"], how="left", validate="one_to_one")[col].values

NIV = {"produccion": "emim_valor_produccion_real", "empleo": "emim_personal_ocupado", "horas": "emim_horas_trabajadas"}
for k, v in NIV.items():
    base["l_" + k] = np.log(base[v].astype(float))
    base["dl1_" + k] = base["l_" + k] - en_fecha("l_" + k, -1)
tabla(pd.DataFrame([dict(variable=f"dl1_{k}", descripcion=f"Δ log mensual de {v}", observaciones=int(base[f"dl1_{k}"].notna().sum())) for k, v in NIV.items()]),
      "Variaciones mensuales (para los rezagos de control)")
E = base.groupby("rama").mip_coef_exportacion_2018.first()
E_P25, E_P75 = E.quantile(0.25), E.quantile(0.75)
ESC_A = 100 * (E_P75 - E_P25)
display(Markdown(f"**Exposición (MIP 2018):** p25 = {E_P25:.3f}, p75 = {E_P75:.3f}. En la especificación A los efectos se reportan para la diferencia p75 − p25."))

**Variables transformadas**

variable,original,descripcion,metodo,observaciones,primer_mes
d12_produccion,emim_valor_produccion_real,Producción real (EMIM),Δ log,7786,2019-01
d12_empleo,emim_personal_ocupado,Personal ocupado (EMIM),Δ log,7826,2019-01
d12_horas,emim_horas_trabajadas,Horas trabajadas (EMIM),Δ log,7826,2019-01
d12_tpu,tpu,TPU (Caldara et al.),Δ log,7826,2019-01
d12_ip_eeuu,eeuu_ip_total,Producción industrial total de EE.UU.,Δ log,7826,2019-01
d12_arancel,eeuu_imp_mx_tasa_efectiva,Tasa arancelaria efectiva de EE.UU. a la rama,diferencia en puntos porcentuales,7735,2019-01
d12_tc,tc_fix,Tipo de cambio FIX,Δ log,7826,2019-01
d12_vix,vix_promedio_mensual,VIX (promedio mensual),Δ log,7826,2019-01
d12_wtui_usa,wtui_usa,WTUI EE.UU.,diferencia en puntos del índice,7826,2019-01
d12_wtui_mundo,wtui_mundo_pib,WTUI mundo,diferencia en puntos del índice,7826,2019-01


**Variaciones mensuales (para los rezagos de control)**

variable,descripcion,observaciones
dl1_produccion,Δ log mensual de emim_valor_produccion_real,8744
dl1_empleo,Δ log mensual de emim_personal_ocupado,8772
dl1_horas,Δ log mensual de emim_horas_trabajadas,8772


**Exposición (MIP 2018):** p25 = 0.158, p75 = 0.618. En la especificación A los efectos se reportan para la diferencia p75 − p25.

### Días hábiles
Lunes a viernes del mes, menos los días de descanso obligatorio de la Ley Federal del Trabajo (art. 74) que caen entre semana: 1 de enero, primer lunes de febrero, tercer lunes de marzo, 1 de mayo, 16 de septiembre, tercer lunes de noviembre, 25 de diciembre y la transmisión del Poder Ejecutivo (1 de diciembre de 2018; 1 de octubre de 2024). Se restan además el **jueves y el viernes santos**: no son obligatorios por ley, pero en la práctica la mayoría de las plantas no trabajan. Es una aproximación: no hay una serie oficial de días laborados por rama.

In [3]:
from datetime import date, timedelta
def pascua(a):
    """Domingo de Pascua (algoritmo anónimo gregoriano)."""
    b, c = divmod(a, 100); d, e = divmod(b, 4); f = (b + 8) // 25; g = (b - f + 1) // 3
    h = (19 * (a % 19) + b - d - g + 15) % 30; i, k = divmod(c, 4); l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a % 19 + 11 * h + 22 * l) // 451; mes = (h + l - 7 * m + 114) // 31; dia = (h + l - 7 * m + 114) % 31 + 1
    return date(a, mes, dia)
def lunes_n(a, mes, n):
    d = date(a, mes, 1)
    return d + timedelta(days=(7 - d.weekday()) % 7 + 7 * (n - 1))
def feriados(a):
    f = {date(a, 1, 1), lunes_n(a, 2, 1), lunes_n(a, 3, 3), date(a, 5, 1), date(a, 9, 16), lunes_n(a, 11, 3), date(a, 12, 25),
         pascua(a) - timedelta(days=3), pascua(a) - timedelta(days=2)}
    f |= {date(2018, 12, 1)} if a == 2018 else set()
    f |= {date(2024, 10, 1)} if a == 2024 else set()
    return f
dias = pd.date_range("2015-01-01", "2027-12-31", freq="D")
fer = set().union(*[feriados(a) for a in range(2015, 2028)])
habil = pd.Series([(d.weekday() < 5) and (d.date() not in fer) for d in dias], index=dias)
DIAS_HABILES = habil.resample("MS").sum().astype(float)
assert DIAS_HABILES.between(17, 23).all()
base["log_dias_habiles"] = np.log(base.mes.map(DIAS_HABILES))
tabla(DIAS_HABILES.loc["2019-01":"2019-12"].rename("días hábiles").rename_axis("mes").reset_index().assign(mes=lambda d: d.mes.dt.strftime("%Y-%m")),
      "Ejemplo: días hábiles en 2019 (Semana Santa en abril)", dec=0)

**Ejemplo: días hábiles en 2019 (Semana Santa en abril)**

mes,días hábiles
2019-01,22
2019-02,19
2019-03,20
2019-04,20
2019-05,22
2019-06,20
2019-07,23
2019-08,22
2019-09,20
2019-10,23


## 2. La sorpresa del TPU
Regresión de pronóstico con toda la historia disponible (1990-2026): log TPU_t sobre 12 rezagos propios, 3 rezagos del log del VIX y **3 rezagos del log de la producción industrial de EE.UU.** La sorpresa es el residuo: la parte del TPU que no se anticipaba con la información del mes anterior. Robustez: error de pronóstico fuera de muestra (ventana creciente desde 2010).

In [4]:
tpu_raw = pd.read_excel(RAIZ / "data/raw/incertidumbre/tpu_caldara_iacoviello.xlsx", sheet_name="TPU_MONTHLY")
tpu_h = tpu_raw.assign(mes=pd.to_datetime(tpu_raw.DATE).dt.to_period("M").dt.to_timestamp()).set_index("mes").TPU.astype(float)
vix_raw = pd.read_csv(RAIZ / "data/raw/controles_macroeconomicos/fred_vix_diario.csv", na_values=["", "."], parse_dates=["observation_date"])
vix_h = vix_raw.dropna().set_index("observation_date").VIXCLS.resample("MS").mean()
ip_raw = pd.read_csv(RAIZ / "data/raw/produccion_estados_unidos/fred_indpro_mensual.csv", parse_dates=["observation_date"])
ip_h = ip_raw.set_index("observation_date").INDPRO.astype(float)
serie_mes = base.drop_duplicates("mes").set_index("mes")
comun = serie_mes[["tpu", "vix_promedio_mensual", "eeuu_ip_total"]].join(pd.DataFrame({"a": tpu_h, "b": vix_h, "c": ip_h}), how="inner")
assert np.allclose(comun.tpu, comun.a) and np.allclose(comun.vix_promedio_mensual, comun.b) and np.allclose(comun.eeuu_ip_total, comun.c)
aviso(f"Las series originales del TPU, VIX y producción industrial de EE.UU. coinciden con la base en los {len(comun)} meses comunes.", "ok")

pron = pd.DataFrame({"ltpu": np.log(tpu_h), "lvix": np.log(vix_h), "lip": np.log(ip_h)})
X_cols = []
for v, n in [("ltpu", 12), ("lvix", 3), ("lip", 3)]:
    for l in range(1, n + 1):
        pron[f"{v}_l{l}"] = pron[v].shift(l)
        X_cols.append(f"{v}_l{l}")
pron = pron.dropna(subset=["ltpu", *X_cols])
reg = sm.OLS(pron.ltpu, sm.add_constant(pron[X_cols])).fit()
sorpresa = reg.resid
oos = {}
for m in pron.index[pron.index >= "2010-01-01"]:
    previo = pron[pron.index < m]
    r_ = sm.OLS(previo.ltpu, sm.add_constant(previo[X_cols])).fit()
    oos[m] = pron.loc[m, "ltpu"] - float(r_.predict(sm.add_constant(pron.loc[[m], X_cols], has_constant="add")).iloc[0])
sorpresa_oos = pd.Series(oos)
tabla(pd.DataFrame([
    dict(prueba="Meses en la regresión de pronóstico", valor=f"{len(pron)} ({pron.index.min():%Y-%m} a {pron.index.max():%Y-%m})"),
    dict(prueba="R²", valor=f"{reg.rsquared:.3f}"),
    dict(prueba="Prueba F: los rezagos de la producción industrial de EE.UU. no ayudan a pronosticar (valor p)",
         valor=f"{reg.f_test(', '.join(f'lip_l{l} = 0' for l in range(1, 4))).pvalue:.3f}"),
    dict(prueba="Autocorrelación de la sorpresa (rezagos 1, 2, 3, 12)", valor=", ".join(f"{sorpresa.autocorr(k):.2f}" for k in (1, 2, 3, 12))),
    dict(prueba="Correlación con la versión fuera de muestra (2018-2026)", valor=f"{sorpresa['2018':].corr(sorpresa_oos['2018':]):.3f}"),
]), "Diagnóstico de la sorpresa")
top = pd.concat([sorpresa.sort_values(ascending=False).head(8), sorpresa.sort_values().head(4)])
tabla(pd.DataFrame({"mes": [m.strftime("%Y-%m") for m in top.index], "sorpresa_log": top.values, "tipo": ["alza"] * 8 + ["baja"] * 4}),
      "Meses con mayor sorpresa (toda la historia)", dec=2)

base = base.merge(pd.DataFrame({"sorpresa_tpu": sorpresa, "sorpresa_tpu_oos": sorpresa_oos}).rename_axis("mes").reset_index(), on="mes", how="left", validate="many_to_one")
serie_mes = base.drop_duplicates("mes").set_index("mes")
ventana = serie_mes.loc[serie_mes.index >= "2018-01-01"]
ESCALA = {}
for c in ["sorpresa_tpu", "sorpresa_tpu_oos", "d12_tpu", "d12_wtui_usa", "d12_wtui_mundo", "d12_wtui_mex", "d12_epu"]:
    ESCALA[c] = ventana[c].std()
    base[c + "_std"] = base[c] / ESCALA[c]
serie_mes = base.drop_duplicates("mes").set_index("mes")
tabla(pd.DataFrame([dict(choque=k, desviacion_estandar_2018_2026=v) for k, v in ESCALA.items()]), "Desviación estándar usada para estandarizar cada choque")

def svg_linea(x, y, titulo, ylab, ancho=860, alto=220, sombra=None):
    L, R, T, B = 60, 20, 30, 34
    lo, hi = np.nanmin(y), np.nanmax(y)
    X = lambda i: L + i / (len(x) - 1) * (ancho - L - R)
    Y = lambda v: T + (hi - v) / (hi - lo) * (alto - T - B)
    p = [f'<svg viewBox="0 0 {ancho} {alto}" style="width:100%;max-width:{ancho}px;font:11px system-ui,sans-serif" role="img">',
         f'<text x="{L}" y="16" style="font-size:13px;font-weight:600">{html.escape(titulo)}</text>']
    if sombra:
        p.append(f'<rect x="{X(sombra[0]):.1f}" y="{T}" width="{X(sombra[1]) - X(sombra[0]):.1f}" height="{alto - T - B}" fill="#e6e5e1"/>')
    for t in np.linspace(lo, hi, 5):
        p.append(f'<line x1="{L}" x2="{ancho - R}" y1="{Y(t):.1f}" y2="{Y(t):.1f}" stroke="#eeeeec"/><text x="{L - 6}" y="{Y(t) + 4:.1f}" text-anchor="end" fill="#7a7974">{t:.1f}</text>')
    p.append(f'<line x1="{L}" x2="{ancho - R}" y1="{Y(0):.1f}" y2="{Y(0):.1f}" stroke="#52514e"/>')
    p.append(f'<polyline points="{" ".join(f"{X(i):.1f},{Y(v):.1f}" for i, v in enumerate(y))}" fill="none" stroke="#2a78d6" stroke-width="2"/>')
    for i, (xx, v) in enumerate(zip(x, y)):
        p.append(f'<circle cx="{X(i):.1f}" cy="{Y(v):.1f}" r="6" fill="transparent"><title>{xx}: {v:.2f}</title></circle>')
        if xx.endswith("-01"):
            p.append(f'<text x="{X(i):.1f}" y="{alto - B + 15}" text-anchor="middle" fill="#7a7974">{xx[:4]}</text>')
    p.append(f'<text x="14" y="{T + (alto - T - B) / 2}" transform="rotate(-90 14 {T + (alto - T - B) / 2})" text-anchor="middle" fill="#7a7974">{ylab}</text></svg>')
    return "".join(p)

PANDEMIA = (pd.Timestamp("2020-03-01"), pd.Timestamp("2021-06-01"))
s2 = serie_mes.sorpresa_tpu_std.loc["2018-01-01":].dropna()
ip2 = [i for i, m in enumerate(s2.index) if PANDEMIA[0] <= m <= PANDEMIA[1]]
display(HTML(svg_linea([m.strftime("%Y-%m") for m in s2.index], s2.values, "Sorpresa del TPU, estandarizada, 2018-2026 (gris: pandemia)",
                       "desv. estándar", sombra=(ip2[0], ip2[-1]))))

**Diagnóstico de la sorpresa**

prueba,valor
Meses en la regresión de pronóstico,437 (1990-04 a 2026-08)
R²,0.882
Prueba F: los rezagos de la producción industrial de EE.UU. no ayudan a pronosticar (valor p),0.790
"Autocorrelación de la sorpresa (rezagos 1, 2, 3, 12)","-0.00, -0.00, -0.00, -0.00"
Correlación con la versión fuera de muestra (2018-2026),0.969


**Meses con mayor sorpresa (toda la historia)**

mes,sorpresa_log,tipo
2018-03,1.43,alza
2024-11,1.06,alza
2025-04,0.84,alza
2016-11,0.81,alza
2013-05,0.69,alza
2019-05,0.65,alza
2015-05,0.63,alza
1993-11,0.62,alza
2013-04,-1.00,baja
2021-02,-0.74,baja


**Desviación estándar usada para estandarizar cada choque**

choque,desviacion_estandar_2018_2026
sorpresa_tpu,0.320
sorpresa_tpu_oos,0.361
d12_tpu,0.855
d12_wtui_usa,0.263
d12_wtui_mundo,"13,192.906"
d12_wtui_mex,0.283
d12_epu,0.695


## 3. Estimador e inferencia por permutación

**Estimación.** Para cada horizonte: (1) se eliminan los efectos fijos (rama × mes del calendario, y mes en A) por proyecciones alternadas, porque el panel no está balanceado; (2) se eliminan los controles (Frisch-Waugh-Lovell); (3) β_h sale de regresar la variable dependiente sobre el choque y sus 3 rezagos. Es idéntico a MCO con todo junto.

**Inferencia por permutación.** La pregunta es: ¿qué tan grande sale β_h si la sorpresa no tuviera nada que ver con la actividad? Para responderla se **reordenan al azar los meses de la sorpresa** (la misma serie, otras fechas) y se vuelve a estimar, 1,000 veces. El **valor p** es la proporción de permutaciones con |β| al menos tan grande como el estimado. La banda gris de las gráficas es el rango 5-95% de esas β falsas: un punto fuera de la banda es significativo al 10%. Es válida si, sin efecto, cualquier orden de la sorpresa es igual de probable; la sorpresa casi no tiene autocorrelación (sección 2), así que el supuesto es razonable. Se usan **las mismas 1,000 permutaciones en todos los horizontes**, lo que permite pruebas conjuntas:

- **Placebo conjunto:** ¿los 8 horizontes h = −12 a −5 son, juntos, distintos de cero?
- **Efecto conjunto:** ¿los 13 horizontes h = 0 a 12 son, juntos, distintos de cero?

El estadístico conjunto es la suma de β_h² / varianza de las β_h falsas, y su valor p se obtiene con las mismas permutaciones.

**Driscoll-Kraay** (rezago máximo 12 + |h|) se reporta solo como referencia.

In [5]:
import scipy.sparse as sp
H_PRE, H_POST = list(range(-12, -4)), list(range(0, 13))
L_REZ, N_PERM = 3, 1000
RNG = np.random.default_rng(20261001)
MESES_S = pd.date_range(base.mes.min() - pd.DateOffset(months=L_REZ), base.mes.max(), freq="MS")
PERM = np.array([RNG.permutation(len(MESES_S)) for _ in range(N_PERM)])
POS_MES = pd.Series(np.arange(len(MESES_S)), index=MESES_S)

SERIES_CHOQUE = {"sorpresa_tpu_std": sorpresa / ESCALA["sorpresa_tpu"], "sorpresa_tpu_oos_std": sorpresa_oos / ESCALA["sorpresa_tpu_oos"]}
def serie_choque(nombre):
    s = SERIES_CHOQUE[nombre].reindex(MESES_S)
    assert s.notna().all(), f"{nombre} tiene meses sin dato"
    return s.values

def proyector_fe(grupos, tol=1e-10, max_iter=2000):
    """Devuelve una función que quita las medias de cada sistema de grupos (proyecciones alternadas)."""
    mats = []
    for g in grupos:
        codigos, _ = pd.factorize(g)
        G = sp.csr_matrix((np.ones(len(codigos)), (np.arange(len(codigos)), codigos)))
        mats.append((G, np.asarray(G.sum(0)).ravel()))
    def f(X):
        X = np.array(X, dtype=float, copy=True)
        for _ in range(max_iter):
            cambio = 0.0
            for G, n in mats:
                medias = (G.T @ X) / n[:, None]
                X -= G @ medias
                cambio = max(cambio, np.abs(medias).max())
            if len(mats) == 1 or cambio < tol:
                break
        return X
    return f

def controles(espec):
    """Controles que no dependen de h ni del choque."""
    E = base.mip_coef_exportacion_2018.values if espec == "A" else np.ones(len(base))
    W = {}
    for k in NIV:
        for l in range(1, L_REZ + 1):
            W[f"dl1_{k}_l{l}"] = en_fecha(f"dl1_{k}", -l)
    for l in range(0, L_REZ + 1):
        W[f"arancel_l{l}"] = en_fecha("d12_arancel", -l)
        W[f"ip_eeuu_l{l}"] = en_fecha("d12_ip_eeuu", -l) * E
    for l in range(1, L_REZ + 1):
        W[f"tc_l{l}"] = en_fecha("d12_tc", -l) * E
        W[f"vix_l{l}"] = en_fecha("d12_vix", -l) * E
    return pd.DataFrame(W), E
CONTROLES = {e: controles(e) for e in ("A", "B")}

POS = {l: (base.mes - pd.DateOffset(months=l)).map(POS_MES).fillna(-1).astype(int).values for l in range(L_REZ + 1)}
_CACHE = {}
def preparar_h(y, espec, excluir_pandemia, h):
    """Datos de un horizonte sin efectos fijos ni controles: y residual (yt) y la matriz MB que convierte la serie del choque en sus 4 columnas."""
    clave = (y, espec, excluir_pandemia, h)
    if clave in _CACHE:
        return _CACHE[clave]
    W0, E = CONTROLES[espec]
    d = W0.copy()
    if espec == "B":
        d["dias_habiles"] = (base.mes + pd.DateOffset(months=h)).map(np.log(DIAS_HABILES)).values - (base.mes - pd.DateOffset(months=1)).map(np.log(DIAS_HABILES)).values
    yh = en_fecha("l_" + y, h) - en_fecha("l_" + y, -1)
    ok = d.notna().all(axis=1).values & ~np.isnan(yh) & np.all([POS[l] >= 0 for l in POS], axis=0)
    if excluir_pandemia:
        ini_ = base.mes + pd.DateOffset(months=min(h, -1))
        fin_ = base.mes + pd.DateOffset(months=max(h, 0))
        ok &= ~((ini_ <= PANDEMIA[1]) & (fin_ >= PANDEMIA[0])).values
    idx = np.flatnonzero(ok)
    m = base.iloc[idx]
    fe = proyector_fe([m.rama + "_" + m.mes.dt.month.astype(str)] + ([m.mes] if espec == "A" else []))
    # Columnas del choque como función lineal de la serie mensual: X_l = B_l @ s, con B_l[r, mes(r) − l] = E_r
    Bs = [sp.csr_matrix((E[idx], (np.arange(len(idx)), POS[l][idx])), shape=(len(idx), len(MESES_S))).toarray() for l in range(L_REZ + 1)]
    k = d.shape[1]
    bloque_ = fe(np.column_stack([yh[idx], d.values[idx], *Bs]))
    yt, Wt = bloque_[:, 0], bloque_[:, 1:1 + k]
    MB = bloque_[:, 1 + k:].reshape(len(idx), L_REZ + 1, len(MESES_S)).transpose(1, 0, 2)
    U, S, _ = np.linalg.svd(Wt, full_matrices=False)
    Q = U[:, S > 1e-9 * S[0]]
    yt = yt - Q @ (Q.T @ yt)
    MB = MB - np.einsum("nk,lkt->lnt", Q, np.einsum("nk,lnt->lkt", Q, MB))
    meta = dict(n=len(idx), ramas=m.rama.nunique(), meses=m.mes.nunique(), desde=m.mes.min().strftime("%Y-%m"), hasta=m.mes.max().strftime("%Y-%m"),
                t_idx=m.mes.rank(method="dense").astype(int).values)
    _CACHE[clave] = (yt, MB, meta)
    return _CACHE[clave]

def betas_perm(yt, MB, s, perm):
    bp = []
    for c in range(0, len(perm), 250):
        Xp = np.einsum("lnt,td->dnl", MB, s[perm[c:c + 250]].T)
        bp.append(np.linalg.solve(np.einsum("dnl,dnk->dlk", Xp, Xp), np.einsum("dnl,n->dl", Xp, yt)[..., None])[:, 0, 0])
    return np.concatenate(bp)

def estimar_lp(y="produccion", choque="sorpresa_tpu_std", espec="A", excluir_pandemia=False):
    s = serie_choque(choque)
    filas, perms = [], {}
    for h in H_PRE + H_POST:
        yt, MB, meta = preparar_h(y, espec, excluir_pandemia, h)
        X = np.einsum("lnt,t->nl", MB, s)
        beta = np.linalg.lstsq(X, yt, rcond=None)[0][0]
        dk = sm.OLS(yt, X).fit(cov_type="nw-groupsum", cov_kwds={"time": meta["t_idx"], "maxlags": 12 + abs(h)})
        bp = perms[h] = betas_perm(yt, MB, s, PERM)
        filas.append(dict(h=h, beta=beta, p_perm=float(np.mean(np.abs(bp) >= abs(beta))), nulo_p05=np.quantile(bp, 0.05), nulo_p95=np.quantile(bp, 0.95),
                          se_dk=dk.bse[0], p_dk=float(dk.pvalues[0]), **{k: v for k, v in meta.items() if k != "t_idx"}))
    r = pd.DataFrame(filas)
    escala = ESC_A if espec == "A" else 100.0
    r["efecto"], r["nulo_p05"], r["nulo_p95"], r["se_dk_efecto"] = r.beta * escala, r.nulo_p05 * escala, r.nulo_p95 * escala, r.se_dk * escala
    def conjunta(hs):
        if not hs:
            return np.nan
        P = np.column_stack([perms[h] for h in hs]); v = P.var(axis=0)
        real = np.sum(r.set_index("h").loc[hs, "beta"].values ** 2 / v)
        return float(np.mean((P ** 2 / v).sum(axis=1) >= real))
    return r.assign(variable=y, choque=choque, especificacion=espec, muestra="sin pandemia" if excluir_pandemia else "completa",
                    p_placebo_conjunto=conjunta(H_PRE), p_efecto_conjunto=conjunta(H_POST))

# Verificación: el estimador por bloques coincide con MCO directo con variables dicótomas (A, producción, h = 3)
_r = estimar_lp("produccion", "sorpresa_tpu_std", "A", False)
_W, _E = CONTROLES["A"]
_d = _W.copy(); _d["y_h"] = en_fecha("l_produccion", 3) - en_fecha("l_produccion", -1)
for l in range(L_REZ + 1):
    _d[f"s_l{l}"] = (base.mes - pd.DateOffset(months=l)).map(SERIES_CHOQUE["sorpresa_tpu_std"]).values * _E
_d = _d.assign(rama=base.rama, mes=base.mes).dropna()
_X = pd.get_dummies(_d.rama + "_" + _d.mes.dt.month.astype(str), drop_first=True, dtype=float).join(pd.get_dummies(_d.mes.dt.strftime("%Y%m"), drop_first=True, dtype=float))
_X = pd.concat([_d[[f"s_l{l}" for l in range(L_REZ + 1)] + list(_W.columns)], _X], axis=1)
_b = sm.OLS(_d.y_h.values, sm.add_constant(_X.values)).fit().params[1]
assert abs(_b - _r.set_index("h").beta[3]) < 1e-6 * max(1, abs(_b)), (_b, _r.set_index("h").beta[3])
aviso(f"Verificación OK: el estimador coincide con MCO con variables dicótomas (A, producción, h = 3: β = {_b:.6f}; n = {len(_d):,}).", "ok")

### Gráficas
Línea azul: β_h (en %). **Banda gris:** rango 5-95% de las β obtenidas con sorpresas permutadas (lo que se vería sin efecto). **Punto relleno:** valor p de permutación < 0.10. A la izquierda de la línea punteada, los placebos.

In [6]:
def marcas(ymin, ymax, n=5):
    paso = (ymax - ymin) / n
    mag = 10 ** np.floor(np.log10(paso))
    paso = min((k * mag for k in (1, 2, 2.5, 5, 10) if k * mag >= paso), default=10 * mag)
    return np.arange(np.floor(ymin / paso) * paso, ymax + paso / 2, paso), paso

def svg_irf(r, titulo, ancho=430, alto=250, color="#2a78d6"):
    L_, R_, T_, B_ = 52, 14, 46, 34
    ticks, paso = marcas(min(r.efecto.min(), r.nulo_p05.min(), 0), max(r.efecto.max(), r.nulo_p95.max(), 0))
    y0, y1 = ticks[0], ticks[-1]
    X = lambda h: L_ + (h + 12) / 24 * (ancho - L_ - R_)
    Y = lambda v: T_ + (y1 - v) / (y1 - y0) * (alto - T_ - B_)
    dec = next(k for k in range(4) if abs(round(paso, k) - paso) < 1e-9)
    p = [f'<svg viewBox="0 0 {ancho} {alto}" style="width:100%;font:11px system-ui,sans-serif" role="img" aria-label="{html.escape(titulo)}">',
         f'<text x="{L_}" y="14" style="font-size:12px;font-weight:600;fill:#0b0b0b">{html.escape(titulo)}</text>',
         f'<text x="{L_}" y="30" style="fill:#52514e">placebo conjunto p = {r.p_placebo_conjunto.iloc[0]:.2f} · efecto conjunto p = {r.p_efecto_conjunto.iloc[0]:.2f}</text>']
    for t in ticks:
        p.append(f'<line x1="{L_}" x2="{ancho - R_}" y1="{Y(t):.1f}" y2="{Y(t):.1f}" stroke="#eeeeec"/><text x="{L_ - 6}" y="{Y(t) + 4:.1f}" text-anchor="end" fill="#7a7974">{round(t, dec) + 0.0:.{dec}f}</text>')
    p.append(f'<text x="12" y="{T_ + (alto - T_ - B_) / 2:.0f}" transform="rotate(-90 12 {T_ + (alto - T_ - B_) / 2:.0f})" text-anchor="middle" fill="#7a7974">%</text>')
    for tramo in (r[r.h < 0], r[r.h >= 0]):
        pts = " ".join(f"{X(h):.1f},{Y(v):.1f}" for h, v in zip(tramo.h, tramo.nulo_p95)) + " " + " ".join(f"{X(h):.1f},{Y(v):.1f}" for h, v in zip(tramo.h[::-1], tramo.nulo_p05[::-1]))
        p.append(f'<polygon points="{pts}" fill="#8a8984" fill-opacity="0.22"/>')
    p.append(f'<line x1="{L_}" x2="{ancho - R_}" y1="{Y(0):.1f}" y2="{Y(0):.1f}" stroke="#52514e"/>')
    p.append(f'<line x1="{X(-2.5):.1f}" x2="{X(-2.5):.1f}" y1="{T_}" y2="{alto - B_}" stroke="#52514e" stroke-dasharray="3 3"/>')
    for tramo, op in ((r[r.h < 0], 0.55), (r[r.h >= 0], 1.0)):
        p.append(f'<polyline points="{" ".join(f"{X(h):.1f},{Y(v):.1f}" for h, v in zip(tramo.h, tramo.efecto))}" fill="none" stroke="{color}" stroke-opacity="{op}" stroke-width="2"/>')
    for _, f in r.iterrows():
        relleno = color if f.p_perm < 0.10 else "white"
        p.append(f'<circle cx="{X(f.h):.1f}" cy="{Y(f.efecto):.1f}" r="3.2" fill="{relleno}" stroke="{color}" stroke-width="1.5"/>'
                 f'<circle cx="{X(f.h):.1f}" cy="{Y(f.efecto):.1f}" r="9" fill="transparent"><title>h={f.h}: {f.efecto:.2f}% · p permutación = {f.p_perm:.3f} · '
                 f'banda nula {f.nulo_p05:.2f} a {f.nulo_p95:.2f} · p Driscoll-Kraay = {f.p_dk:.3f}</title></circle>')
    for h in (-12, -9, -6, 0, 3, 6, 9, 12):
        p.append(f'<text x="{X(h):.1f}" y="{alto - B_ + 15}" text-anchor="middle" fill="#7a7974">{h}</text>')
    p.append(f'<text x="{(L_ + ancho - R_) / 2}" y="{alto - 4}" text-anchor="middle" fill="#7a7974">meses respecto al choque (h)</text></svg>')
    return "".join(p)

resultados = []
def bloque(choque, espec, titulo_bloque):
    display(Markdown(f"### {titulo_bloque}"))
    celdas = []
    for y in NIV:
        for sp_ in (False, True):
            r = estimar_lp(y=y, choque=choque, espec=espec, excluir_pandemia=sp_)
            resultados.append(r)
            celdas.append(svg_irf(r, f"{NOMBRE_Y[y]} · {'sin pandemia' if sp_ else 'muestra completa'}"))
    display(HTML('<div class="irf-grid">' + "".join(celdas) + "</div>"))
NOMBRE_Y = {"produccion": "Producción real", "empleo": "Empleo", "horas": "Horas trabajadas"}

## 4. Especificación A: exposición diferencial (principal)
Diferencia en % entre una rama de exposición alta (p75) y una baja (p25) ante una sorpresa del TPU de 1 desviación estándar.

In [7]:
bloque("sorpresa_tpu_std", "A", "A. Choque: sorpresa del TPU")

### A. Choque: sorpresa del TPU

## 5. Especificación B: efecto promedio (descriptiva)
% para la rama promedio ante una sorpresa de 1 desviación estándar. Se identifica solo con la variación en el tiempo (un solo choque agregado en ~70-90 meses), así que es mucho menos informativa que A.

In [8]:
bloque("sorpresa_tpu_std", "B", "B. Choque: sorpresa del TPU")

### B. Choque: sorpresa del TPU

## 6. Robustez y diagnóstico de la inferencia
**6a. Sorpresa fuera de muestra** (pronóstico con ventana creciente: solo usa información disponible en cada mes), especificación A.

In [9]:
celdas = []
for y in NIV:
    for sp_ in (False, True):
        r = estimar_lp(y=y, choque="sorpresa_tpu_oos_std", espec="A", excluir_pandemia=sp_)
        resultados.append(r.assign(robustez="sorpresa fuera de muestra"))
        celdas.append(svg_irf(r, f"{NOMBRE_Y[y]} · {'sin pandemia' if sp_ else 'muestra completa'}", color="#eb6834"))
display(HTML('<div class="irf-grid">' + "".join(celdas) + "</div>"))

**6b. ¿Por qué no Driscoll-Kraay?** Se toman 200 de las sorpresas permutadas (que por construcción no tienen efecto) y se estima la especificación A de producción sin pandemia con cada una, en 5 horizontes. Con errores bien calibrados, la prueba al 10% debería rechazar en ~10% de los casos.

In [10]:
def tasa_rechazo_dk(y="produccion", espec="A", excluir_pandemia=True, hs=(0, 3, 6, 9, 12), n_sim=200, n_perm=500):
    s = serie_choque("sorpresa_tpu_std")
    filas = []
    for j in range(n_sim):
        falso = s[PERM[-1 - j]]                      # una sorpresa reordenada: sin efecto por construcción
        for h in hs:
            yt, MB, meta = preparar_h(y, espec, excluir_pandemia, h)
            X = np.einsum("lnt,t->nl", MB, falso)
            dk = sm.OLS(yt, X).fit(cov_type="nw-groupsum", cov_kwds={"time": meta["t_idx"], "maxlags": 12 + h})
            bp = betas_perm(yt, MB, falso, PERM[:n_perm])
            filas.append(dict(sim=j, h=h, rechaza_dk=dk.pvalues[0] < 0.10, rechaza_perm=np.mean(np.abs(bp) >= abs(dk.params[0])) < 0.10))
    return pd.DataFrame(filas)
_t = tasa_rechazo_dk()
tabla(pd.DataFrame([dict(prueba="Driscoll-Kraay (rezago 12 + h)", rechazo_al_10=_t.rechaza_dk.mean()),
                    dict(prueba="Permutación", rechazo_al_10=_t.rechaza_perm.mean())]),
      f"Proporción de rechazos con sorpresas falsas ({_t.sim.nunique()} simulaciones × {_t.h.nunique()} horizontes; lo correcto es 0.10)", dec=3)

**Proporción de rechazos con sorpresas falsas (200 simulaciones × 5 horizontes; lo correcto es 0.10)**

prueba,rechazo_al_10
Driscoll-Kraay (rezago 12 + h),0.404
Permutación,0.100


## 7. Resumen y comparación con la especificación anterior
**Resumen** (de 13 horizontes h = 0 a 12): efecto medio, horizontes con valor p de permutación < 0.10 por signo, y las dos pruebas conjuntas. Para comparar se muestra cuántos horizontes serían significativos con Driscoll-Kraay.

In [11]:
res = pd.concat(resultados, ignore_index=True)
res["robustez"] = res.get("robustez", pd.Series(index=res.index, dtype=object)).fillna("")
etiqueta_choque = {"sorpresa_tpu_std": "sorpresa TPU", "sorpresa_tpu_oos_std": "sorpresa TPU (fuera de muestra)"}
post = res[res.h >= 0].assign(neg=lambda d: (d.p_perm < 0.10) & (d.beta < 0), pos=lambda d: (d.p_perm < 0.10) & (d.beta > 0), dk=lambda d: d.p_dk < 0.10)
resumen = (post.assign(choque=post.choque.map(etiqueta_choque), variable=post.variable.map(NOMBRE_Y))
           .groupby(["especificacion", "choque", "variable", "muestra"], sort=False)
           .agg(efecto_medio_h0_12=("efecto", "mean"), efecto_min=("efecto", "min"), efecto_max=("efecto", "max"),
                sig_perm_negativos=("neg", "sum"), sig_perm_positivos=("pos", "sum"), sig_driscoll_kraay=("dk", "sum"),
                p_efecto_conjunto=("p_efecto_conjunto", "first"), p_placebo_conjunto=("p_placebo_conjunto", "first"), meses_h0=("meses", "first")).reset_index())
tabla(resumen, "Resumen (efectos en %; horizontes significativos al 10%)", dec=2, max_filas=200)

**Resumen (efectos en %; horizontes significativos al 10%)**

especificacion,choque,variable,muestra,efecto_medio_h0_12,efecto_min,efecto_max,sig_perm_negativos,sig_perm_positivos,sig_driscoll_kraay,p_efecto_conjunto,p_placebo_conjunto,meses_h0
A,sorpresa TPU,Producción real,completa,1.55,0.65,2.15,0,10,13,0.00,0.72,88
A,sorpresa TPU,Producción real,sin pandemia,0.65,-0.41,1.80,0,5,7,0.04,0.50,71
A,sorpresa TPU,Empleo,completa,-0.06,-0.27,0.11,0,1,1,0.53,0.78,88
A,sorpresa TPU,Empleo,sin pandemia,-0.00,-0.17,0.18,0,0,6,0.61,0.74,71
A,sorpresa TPU,Horas trabajadas,completa,-0.10,-0.51,0.24,0,0,4,0.51,0.24,88
A,sorpresa TPU,Horas trabajadas,sin pandemia,0.04,-0.28,0.33,0,0,8,0.72,0.14,71
B,sorpresa TPU,Producción real,completa,0.61,-1.64,2.30,0,3,2,0.25,0.08,88
B,sorpresa TPU,Producción real,sin pandemia,-0.08,-0.67,0.32,1,0,3,0.45,0.60,71
B,sorpresa TPU,Empleo,completa,-0.11,-0.54,0.11,0,0,0,0.74,0.08,88
B,sorpresa TPU,Empleo,sin pandemia,-0.03,-0.24,0.12,0,0,4,0.75,0.47,71


**Comparación con la especificación anterior** (A, sorpresa del TPU; efecto en %, p75 − p25). La anterior usaba Δ12 como variable dependiente, sin tipo de cambio × exposición, y su significancia era de Driscoll-Kraay. Sus resultados están congelados en `outputs/tables/lp_resultados_especificacion_anterior.csv`.

In [12]:
ant = pd.read_csv(RAIZ / "outputs/tables/lp_resultados_especificacion_anterior.csv", keep_default_na=False, na_values=[""])
ant = ant[(ant.especificacion == "A") & (ant.choque == "sorpresa_tpu_std") & ant.robustez.isna()]
nueva = res[(res.especificacion == "A") & (res.choque == "sorpresa_tpu_std") & (res.robustez == "") & (res.h >= 0)]
comp = nueva.merge(ant[["variable", "muestra", "h", "efecto_pp", "t_dk"]], on=["variable", "muestra", "h"], how="left", validate="one_to_one")
comp["anterior"] = comp.efecto_pp.map("{:.2f}".format) + np.where(comp.t_dk.abs() > 1.645, " *", "")
comp["ahora"] = comp.efecto.map("{:.2f}".format) + np.where(comp.p_perm < 0.10, " *", "")
for y in NIV:
    t_ = comp[comp.variable == y].pivot(index="h", columns="muestra", values=["anterior", "ahora"])
    t_.columns = [f"{a} · {b}" for a, b in t_.columns]
    tabla(t_.reset_index(), f"{NOMBRE_Y[y]}: anterior vs. ahora (* = significativo al 10%: Driscoll-Kraay antes, permutación ahora)")

**Producción real: anterior vs. ahora (* = significativo al 10%: Driscoll-Kraay antes, permutación ahora)**

h,anterior · completa,anterior · sin pandemia,ahora · completa,ahora · sin pandemia
0,-0.14,-0.42 *,0.65 *,0.49
1,0.44,0.05,1.44 *,0.99 *
2,0.66,0.33,2.00 *,1.56 *
3,0.43,0.15,2.11 *,1.80 *
4,0.70,-0.04,1.32 *,0.63
5,1.00 *,0.15,2.15 *,1.40 *
6,0.53,-0.82 *,1.75 *,0.61
7,0.81,-0.64 *,1.22,-0.03
8,1.34 *,-0.14,0.96,0.00
9,0.21,-1.14 *,1.16,-0.41


**Empleo: anterior vs. ahora (* = significativo al 10%: Driscoll-Kraay antes, permutación ahora)**

h,anterior · completa,anterior · sin pandemia,ahora · completa,ahora · sin pandemia
0,0.06,0.08,0.02,0.01
1,0.18 *,0.21 *,0.11 *,0.09
2,0.09,0.11,0.09,0.04
3,-0.00,0.06,0.08,0.10
4,-0.01,0.06,0.09,0.16
5,0.10,0.19,0.05,0.18
6,-0.08,-0.06,-0.04,-0.01
7,-0.10,-0.05,-0.12,0.01
8,-0.02,-0.06,-0.09,-0.11
9,-0.14,-0.15,-0.22,-0.16


**Horas trabajadas: anterior vs. ahora (* = significativo al 10%: Driscoll-Kraay antes, permutación ahora)**

h,anterior · completa,anterior · sin pandemia,ahora · completa,ahora · sin pandemia
0,-0.17,-0.10,0.08,0.11
1,0.09,-0.02,0.15,0.04
2,0.18,0.29 *,0.24,0.23
3,-0.04,-0.01,0.22,0.32
4,-0.16,-0.23,-0.31,-0.22
5,0.28 *,0.33 *,0.02,0.33
6,-0.32,-0.43,0.01,0.25
7,-0.19,-0.31 *,-0.36,-0.20
8,-0.10,0.05,-0.19,0.19
9,-0.36 *,-0.46 *,-0.14,-0.14


In [13]:
res.drop(columns=[c for c in ["se_dk"] if c in res]).to_csv(RAIZ / "outputs/tables/lp_resultados.csv", index=False)
resumen.to_csv(RAIZ / "outputs/tables/lp_resumen.csv", index=False)
pd.DataFrame({"sorpresa_tpu": sorpresa, "sorpresa_tpu_oos": sorpresa_oos}).rename_axis("mes").to_csv(RAIZ / "outputs/tables/sorpresa_tpu.csv")
_t.to_csv(RAIZ / "outputs/tables/lp_diagnostico_driscoll_kraay.csv", index=False)
display(Markdown(f"Guardado en `outputs/tables/`: `lp_resultados.csv` (todos los coeficientes, incluidos placebos), `lp_resumen.csv`, `sorpresa_tpu.csv` y `lp_diagnostico_driscoll_kraay.csv`.  \n"
                 f"**Fin:** {datetime.now():%Y-%m-%d %H:%M:%S} · Python {platform.python_version()}, pandas {pd.__version__}, statsmodels {statsmodels.__version__} · {N_PERM} permutaciones (semilla 20261001)"))

Guardado en `outputs/tables/`: `lp_resultados.csv` (todos los coeficientes, incluidos placebos), `lp_resumen.csv`, `sorpresa_tpu.csv` y `lp_diagnostico_driscoll_kraay.csv`.  
**Fin:** 2026-10-02 00:10:16 · Python 3.11.15, pandas 3.0.6, statsmodels 0.15.0 · 1000 permutaciones (semilla 20261001)